# 11.7 樹狀分支遞迴（Tree Recursion）與數論演算法（費氏數列與輾轉相除法 GCD）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_11-7_tree_recursion_and_number_theory.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**先備知識**：已掌握 11.1 函數定義與呼叫、11.2 函數回傳值、11.4 變數作用域與生命週期，以及 11.6 線性遞迴與呼叫堆疊。

---

### 學習導覽：突破單線思維，探索分支展開與數論的極致美學

在單元 11.6 中，我們掌握了「單線深入、原路折返」的線性遞迴（Linear Recursion）。然而，遞迴真正的威力絕不僅限於一條單行道！

當一個函數在自己的執行過程中，**同時呼叫了兩次或多次自己**時，整個呼叫路徑就會像一棵參天巨木般枝繁葉茂地展開——這就是演算法世界中最壯麗的**樹狀分支遞迴（Tree Recursion）**！

在本單元中，我們將跨越單線思維，深入探討以下 6 個關鍵主題：
1. **11.7.1 雙分支遞迴**：一分二的「遞迴展開樹（Recursion Tree）」物理模型與執行時序。
2. **11.7.2 經典費氏數列（Fibonacci）**：$F(n) = F(n-1) + F(n-2)$ 的雙重 Base Case 與樹狀走訪軌跡。
3. **11.7.3 重疊子問題的認知衝擊**：剖析樹狀遞迴引發的 $O(2^n)$ 指數級運算量爆炸真相。
4. **11.7.4 經典數論：歐幾里得輾轉相除法（GCD）**：兩千年古希臘智慧，短短兩行搞定最大公因數。
5. **11.7.5 快速冪運算遞迴版**：將 $O(N)$ 降至 $O(\log N)$ 的分治思維萌芽。
6. **11.7.6 遞迴與迴圈的終極權衡**：何時該用迴圈（空間 $O(1)$）？何時遞迴思維無可取代？

讓我們踏上這段充滿驚喜的分支探索之旅，領略演算法深邃的美感！

### 11.7.1 雙分支遞迴：一分二的遞迴樹（Recursion Tree）展開圖解

#### 1. 生活故事比喻：迷宮分岔路口的分身術
想像你是一位在迷宮中探索的冒險者。前方的路突然遇到了一個「丁字路口」，道路分成左邊通道與右邊通道。
如果你只有一個人，你必須先走完左邊、退回來、再走右邊；
但如果你擁有神奇的「影分身術」：在路口你召喚出兩位縮小版的分身，一個派去探測左通道、一個派去探測右通道；每個分身走進去如果又遇到分岔路口，就再次分裂成更小的兩個分身！
最後，所有底層分身把探測到的金幣數量往回交給隊長，隊長將「左分身找到的金幣」加上「右分身找到的金幣」，回報給主程式。這就是**雙分支遞迴（Tree Recursion）**！

#### 2. 底層運作機制：一分二的展開時序
當函數內部包含兩次遞迴呼叫時，例如：
```python
def branch(n):
    if n <= 0: return
    branch(n - 1)  # 左分支
    branch(n - 1)  # 右分支
```
在單執行緒的 Python 直譯器中，這**絕不是兩邊同時平行運算**！
直譯器在執行堆疊（Call Stack）上，永遠嚴格遵守**「深度優先（Depth-First）」**的順序：
1. 先一頭鑽進左子樹，一路遞推深入到底層 Base Case。
2. 左分支徹底執行完畢並全部彈出之後，才輪到右分支開始往下鑽。
3. 左右兩邊的答案都算出來後，當前層才將兩者合併並回傳。
整體的呼叫結構在視覺上形成一棵美麗的「二元樹（Binary Tree）」。

#### 3. 初學者常見陷阱：大腦試圖同時追蹤兩邊
初學同學在理解雙分支遞迴時，最容易感到大腦「當機卡死」，因為人類的大腦不擅長同時追蹤兩條平行的路徑。
請記住破解心法：**「相信子問題的合約（Trust the Recursion）」**！
不要試圖在大腦中把幾十個分支全數展開。你只需要堅信：
- 左邊的呼叫 `f(n-1)` 會精準回傳左邊的正確解答；
- 右邊的呼叫 `f(n-1)` 會精準回傳右邊的正確解答；
- 當前層只需要負責把這兩份現成的答案加起來即可！

#### 4. APCS 實戰視野
雙分支遞迴是 APCS 實作題邁向第三級、第四級不可或缺的基石。二元樹前中後序走訪、二分搜尋（Binary Search）、分治演算法（Divide and Conquer）以及回溯枚舉，本質上都是樹狀遞迴的精彩化身。

In [ ]:
# 範例 11.7.1：觀察雙分支遞迴的深度優先展開順序

def tree_explore(label, depth):
    # 縮排視覺化呈現呼叫深度
    indent = "  " * depth
    print(f"{indent}👉 抵達分支 [{label}] (深度 {depth})")
    
    # Base Case: 深度達到 2 停止
    if depth >= 2:
        print(f"{indent}  🛑 分支 [{label}] 觸發基底停止")
        return
        
    # 先深入左分支
    tree_explore(label + "-左", depth + 1)
    
    # 左分支完全結束後，才深入右分支
    tree_explore(label + "-右", depth + 1)
    
    print(f"{indent}👈 分支 [{label}] 任務完成，返回上一層")

# 主程式啟動雙分支探索
print("=== 雙分支遞迴展開開始 ===")
tree_explore("根節點", 0)
print("=== 探索圓滿結束 ===")

In [ ]:
# ==========================================
# [3] Code 填空題 11.7.1
# 任務說明：
# 某位同學正在設計一個二元樹葉節點計數器 `count_leaves(depth)`。
# 樹的分裂規則：每一層一分為二，直到 depth == 0 時該節點為葉節點（計為 1）。
# 請補齊程式碼中的 `___`：
# 1. Base Case：若 depth == 0 回傳 1。
# 2. Recursive Case：將左分支與右分支的葉節點數量相加。
# ==========================================

def count_leaves(depth):
    # 提示：最底層葉節點數量為 1
    if depth == 0:
        return ___
        
    # 提示：左子樹與右子樹規模均縮小 1 層，將兩者結果相加
    left_leaves = count_leaves(depth - 1)
    right_leaves = count_leaves(___)
    return left_leaves + right_leaves

# 主程式測試 depth = 3 的葉節點總數（預期 2^3 = 8）
ans = count_leaves(3)
print(f"深度 3 的二元樹葉節點總數: {ans}")  # 預期輸出: 8

In [ ]:
# ==========================================
# [4] Code 練習題 11.7.1
# 任務說明：
# 請設計一個雙分支印出對稱符號樹的函數 `branch_shapes(n)`：
# 1. 接收非負整數 n 代表層數。
# 2. 若 n == 0，印出 "#" 並結束。
# 3. 若 n > 0：
#    - 印出 "["。
#    - 遞迴呼叫 `branch_shapes(n - 1)`（左邊）。
#    - 印出 "|"（中央隔板）。
#    - 遞迴呼叫 `branch_shapes(n - 1)`（右邊）。
#    - 印出 "]"。
# 4. 所有輸出不可換行（使用 end=""），最外層呼叫完畢後印出換行。
#
# 【公開測試資料 1】
# 呼叫：branch_shapes(1)
# 預期輸出：
# [#|#]
#
# 【公開測試資料 2】
# 呼叫：branch_shapes(2)
# 預期輸出：
# [[#|#]|[#|#]]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.7.1
# 任務說明：
# 請設計一個雙分支金幣累加樹 `tree_coins(level)`：
# 1. 當 level == 1 時（最底層），回傳金幣數 5。
# 2. 當 level > 1 時：
#    - 當前節點自身攜帶 level 枚金幣。
#    - 加上左子樹 `tree_coins(level - 1)` 與右子樹 `tree_coins(level - 1)` 的金幣總和。
# 3. 主程式分別計算 level = 1, level = 2, level = 3 的整棵樹金幣總額並印出。
# 4. 在註解中寫下 level = 3 時的數學展開式。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

### 11.7.2 經典費氏數列（Fibonacci）：$F(n) = F(n-1) + F(n-2)$ 的樹狀執行流程

#### 1. 生活故事比喻：大自然中神奇的兔子家族
13 世紀義大利數學家費波那契（Fibonacci）曾提出一個著名的兔子繁殖問題：
假設一對新生的小兔子，需要一個月長大成成熟的大兔子；成熟的大兔子每個月都會生出一對新小兔子，且兔子永遠不會死去。
- 第 0 個月：0 對兔子
- 第 1 個月：1 對兔子
- 第 2 個月：1 對兔子（長大為大兔子）
- 第 3 個月：2 對兔子（大兔子生出 1 對小兔子）
- 第 4 個月：3 對兔子……
每個月份的兔子數量，正好等於「前一個月的兔子數量（上月活下來的）」加上「前兩個月的兔子數量（已經成熟並生下小兔子的）」！
數列為：$0, 1, 1, 2, 3, 5, 8, 13, 21, 34 \dots$

#### 2. 底層運作機制：雙重 Base Case 與樹狀遞迴
費氏數列的數學遞迴定義：
$$F(n) = \begin{cases} 0 & \text{if } n = 0 \quad (\text{Base Case 1}) \\ 1 & \text{if } n = 1 \quad (\text{Base Case 2}) \\ F(n-1) + F(n-2) & \text{if } n \ge 2 \quad (\text{Recursive Case}) \end{cases}$$

觀察計算 $F(4)$ 的展開歷程：
```
              F(4)
            /      \
         F(3)        F(2)
        /    \      /    \
      F(2)   F(1)  F(1)   F(0)
     /    \
   F(1)   F(0)
```
- **雙重基底**：因為每一步需要用到前兩項，所以 Base Case 必須同時涵蓋 $n=0$ 與 $n=1$ 兩個端點！
- **計算順序**：先一路向下算完最左邊的 $F(2) \rightarrow F(1) \rightarrow F(0)$，得到 $F(2)=1$；再回頭加上 $F(1)=1$，得到 $F(3)=2$；最後再算右半邊的 $F(2)=1$，兩者相加得到 $F(4) = 2 + 1 = 3$！

#### 3. 初學者常見陷阱：Base Case 遺漏 0
初學同學常順手只寫 `if n <= 1: return 1`。
這會導致致命的邏輯錯誤：
這樣寫的話，$F(0)$ 會算出 1，$F(1)$ 也是 1，導致整條數列全部位移錯位（$F(2)$ 變成 2，$F(3)$ 變成 3……）！
請務必精準定義：
```python
if n == 0: return 0
if n == 1: return 1
```
或者合併為防禦性寫法：`if n <= 0: return 0; if n == 1: return 1`。

#### 4. APCS 實戰視野
費氏數列是 APCS 觀念題中考察「樹狀遞迴呼叫次數」與「回傳值追蹤」的黃金標準題。熟練手繪二元展開樹，能讓你在考場上面對這類題型時秒殺拿分。

In [ ]:
# 範例 11.7.2：樸素費氏數列的遞迴實作與呼叫追蹤

call_count = 0

def fib(n):
    global call_count
    call_count += 1
    
    # 雙重 Base Case
    if n <= 0:
        return 0
    if n == 1:
        return 1
        
    # 雙分支 Recursive Case
    return fib(n - 1) + fib(n - 2)

# 測試前 7 項費氏數列數值
print("=== 費氏數列前 8 項計算 ===")
for i in range(8):
    call_count = 0
    val = fib(i)
    print(f"F({i}) = {val:2d} (共呼叫了 {call_count:2d} 次函數)")

In [ ]:
# ==========================================
# [3] Code 填空題 11.7.2
# 任務說明：
# 某位同學正在實作費氏數列的遞迴計算。
# 請補齊程式碼中的 `___`：
# 1. 補齊雙重 Base Case（n <= 0 回傳 0，n == 1 回傳 1）。
# 2. 補齊雙分支相加公式。
# ==========================================

def get_fib(n):
    # 提示：第 0 項為 0
    if n <= 0:
        return ___
    # 提示：第 1 項為 1
    if n == 1:
        return ___
        
    # 提示：前一項加上前兩項
    return get_fib(n - 1) + get_fib(___)

# 主程式測試 F(6)（預期 8）
ans = get_fib(6)
print(f"F(6) 的計算結果: {ans}")  # 預期輸出: 8

In [ ]:
# ==========================================
# [4] Code 練習題 11.7.2
# 任務說明：
# 請設計一個「三分支變形數列」函數 `trib(n)`（類似 Tribonacci 數列簡化版）：
# 規定：
# 1. 當 n <= 0 時回傳 0。
# 2. 當 n == 1 時回傳 1。
# 3. 當 n == 2 時回傳 1。
# 4. 當 n >= 3 時，每一項等於前三項的總和：`trib(n-1) + trib(n-2) + trib(n-3)`。
#
# 【公開測試資料 1】
# 呼叫：trib(3)
# 預期輸出：
# trib(3) = 2  (0 + 1 + 1)
#
# 【公開測試資料 2】
# 呼叫：trib(5)
# 預期輸出：
# trib(5) = 7  (數列: 0, 1, 1, 2, 4, 7)
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.7.2
# 任務說明：
# 請設計一個「爬樓梯問題（Climbing Stairs）」的遞迴解法：
# 題目情境：你要爬上一座共有 n 階的樓梯。每次你只能選擇爬 1 階或爬 2 階。
# 請問爬到第 n 階共有多少種不同的走法？
# 1. 遞迴關係思考：
#    - 要到達第 n 階，最後一步只能是從第 n-1 階跨 1 步上來，或是從第 n-2 階跨 2 步上來！
#    - 因此：`ways(n) = ways(n-1) + ways(n-2)`！
# 2. Base Case 思考：
#    - 當 n == 1 時只有 1 種方法（跨 1 步）。
#    - 當 n == 2 時有 2 種方法（跨兩次 1 步，或直接跨 2 步）。
# 3. 撰寫 `count_stair_ways(n)` 並印出 n = 1 到 n = 6 的方法數，感受它與費氏數列的神奇連結。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

### 11.7.3 重疊子問題（Overlapping Subproblems）的感性認知與運算量爆炸分析

#### 1. 生活故事比喻：失憶的老學者與無止境的重複計算
想像有一位精通算術的老學者坐在書桌前計算 $F(5)$。
他花了整整 5 分鐘，辛辛苦苦在草稿紙上把 $F(3)$ 算了出來，得到答案 2。
接著，為了算出 $F(5)$ 的另一半，他又需要用到 $F(3)$。但他完全沒有把剛才的答案記在小本子上記錄下來，而是像失憶了一樣，把剛才算過的所有步驟從頭到尾**一模一樣地又重算了一遍**！
更可怕的是：在算 $F(3)$ 的過程中，他又重複算了兩次 $F(2)$、三次 $F(1)$！
這種「同一個子問題被完全無意義地重複計算成千上萬次」的現象，就是演算法中最經典的**「重疊子問題（Overlapping Subproblems）」**！

#### 2. 底層運作機制：指數級複雜度 $O(2^n)$ 的災難性爆炸
請仔細觀察回顧 11.7.2 中 $F(4)$ 的展開樹：
- 為了算 $F(4)$，$F(2)$ 被完整計算了 **2 次**！
- 如果要算 $F(5)$，$F(3)$ 被算 2 次，$F(2)$ 被算 **3 次**！
- 如果要算 $F(30)$，函數總呼叫次數高達 **269 萬次**！
- 如果要算 $F(40)$，呼叫次數暴增到超過 **3.3 億次**！在普通電腦上執行會直接卡死數十秒！
- 如果要算 $F(50)$，呼叫次數將達到 $200 兆次$，就算全宇宙最強大的超級電腦也要算上幾天幾夜！
這就是**時間複雜度 $O(2^n)$** 的恐怖威力！每增加一階，計算量就直接翻倍！

#### 3. 初學者的重大頓悟：遞迴不是永遠的神
很多初學者在學會遞迴後，會盲目崇拜遞迴，覺得用迴圈寫程式「很土」，用遞迴「很潮」。
但在這個例子中，事實狠狠上了一課：
- **樸素樹狀遞迴**：算 $F(40)$ 需要呼叫 3 億次函數，耗時十幾秒；
- **傳統 for 迴圈（迭代）**：只要 40 次簡單的加法（`a, b = b, a + b`），耗時不到 **0.00001 秒**，快了數百萬倍！

#### 4. APCS 實戰視野
理解重疊子問題，是跨向 APCS 實作第四級「動態規劃（Dynamic Programming, DP）」與「記憶化搜尋（Memoization）」的關鍵鑰匙。高分選手看到重疊子問題，第一反應就是：「把算過的答案存進字典或串列，下次直接查表，就能把 $O(2^n)$ 瞬間降為極速的 $O(n)$！」

In [ ]:
# 範例 11.7.3：實測樸素遞迴的呼叫次數爆炸現象

call_counter = 0

def slow_fib(n):
    global call_counter
    call_counter += 1
    if n <= 0: return 0
    if n == 1: return 1
    return slow_fib(n - 1) + slow_fib(n - 2)

def fast_loop_fib(n):
    """傳統 for 迴圈版本：線性 O(n) 極速"""
    if n <= 0: return 0
    if n == 1: return 1
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("=== 觀察呼叫次數隨著 n 呈幾何級數暴增 ===")
for test_n in [5, 10, 15, 20]:
    call_counter = 0
    ans = slow_fib(test_n)
    print(f"n = {test_n:2d} -> 答案: {ans:4d} | 遞迴呼叫次數: {call_counter:6d} 次")

print("\n=== 對照組：用迴圈計算 n = 50 ===")
ans_50 = fast_loop_fib(50)
print(f"n = 50 -> 迴圈瞬間秒殺！答案: {ans_50}")

In [ ]:
# ==========================================
# [3] Code 填空題 11.7.3
# 任務說明：
# 某位同學想用字典實作最基礎的「記憶化查表（Memoization）」，
# 徹底消滅費氏數列的重複計算！
# 請補齊程式碼中的 `___`：
# 1. 在計算前先檢查 n 是否已經存在於 memo 字典中。
# 2. 計算完畢後，將答案存入 memo[n] 並回傳。
# ==========================================

memo = {}
call_count_memo = 0

def memo_fib(n):
    global call_count_memo
    call_count_memo += 1
    
    # 提示：查表法！若之前算過，直接回傳記憶中的答案
    if n in memo:
        return memo[___]
        
    # Base Case
    if n <= 0:
        return 0
    if n == 1:
        return 1
        
    # 提示：計算答案並存入字典備忘錄
    result = memo_fib(n - 1) + memo_fib(n - 2)
    memo[n] = ___
    return result

# 主程式測試計算 n = 20
ans = memo_fib(20)
print(f"F(20) = {ans}，呼叫次數大幅縮減為: {call_count_memo} 次！")
# 預期：原本需要 21891 次呼叫，查表優化後僅需 39 次！

In [ ]:
# ==========================================
# [4] Code 練習題 11.7.3
# 任務說明：
# 請設計一個能統計「特定數值被重複計算了幾次」的函數：
# 1. 撰寫樸素遞迴 `count_target_calls(n, target)`。
# 2. 全域變數 `target_hit = 0`。
# 3. 每當進入函數時，若當前的參數 n 等於 target，則將 `target_hit` 加 1。
# 4. 統計計算 slow_fib(6) 的過程中，`F(2)` 究竟被重複呼叫了幾次。
#
# 【公開測試資料 1】
# 呼叫：計算 F(5) 中 F(2) 的呼叫次數
# 預期輸出：
# 計算 F(5) 時，F(2) 被重複呼叫了: 3 次
#
# 【公開測試資料 2】
# 呼叫：計算 F(6) 中 F(2) 的呼叫次數
# 預期輸出：
# 計算 F(6) 時，F(2) 被重複呼叫了: 5 次
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.7.3
# 任務說明：
# 請進行一場「樸素遞迴 vs 記憶化遞迴」的極限對抗：
# 1. 分別使用單元中學過的 `slow_fib(n)` 與 `memo_fib(n)` 計算 n = 25。
# 2. 同時統計兩者在計算 n = 25 時各自的總呼叫次數。
# 3. 輸出對比報表，算出記憶化優化節省了多少倍的函數呼叫次數。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

### 11.7.4 經典數論：輾轉相除法求最大公因數（GCD）之極簡遞迴實現

#### 1. 生活故事比喻：古希臘建築師的完美地磚拼貼
在兩千多年前的古希臘，數學家歐幾里得（Euclid）在雅典城幫神廟鋪設地面。
神廟的長方形地面長 48 公尺、寬 18 公尺。建築師想要用「最大的正方形地磚」完全鋪滿地面，一塊地磚都不能切碎！
歐幾里得的方法非常天才：
他先拿最大的 $18 \times 18$ 正方形地磚鋪進去，鋪了兩塊（佔去 $18 \times 2 = 36$ 公尺），此時地面剩下了一塊 $18 \times 12$ 的長方形空地（這就是取餘數：$48 \% 18 = 12$）。
歐幾里得說：「任何能鋪滿 $18 \times 18$ 與剩下空地的正方形地磚，其邊長必定也能整除 48 與 18！因此，**求 48 與 18 的最大公因數，完全等價於求 18 與 12 的最大公因數！**」
接著再用 $12 \times 12$ 地磚鋪進 $18 \times 12$，剩下 $12 \times 6$；
最後用 $6 \times 6$ 正好整除完全鋪滿（餘數為 0）！最大的地磚邊長就是 **6**！

#### 2. 底層運作機制：短短兩行的極速數論黑科技
這就是歷史上最著名的「歐幾里得輾轉相除法（Euclidean Algorithm）」。
其數學遞迴方程式簡潔得不可思議：
$$\gcd(a, b) = \begin{cases} a & \text{if } b = 0 \quad (\text{Base Case：當餘數歸零時，當前的 a 即為答案}) \\ \gcd(b, a \% b) & \text{if } b > 0 \quad (\text{Recursive Case：除數變被除數，餘數變除數}) \end{cases}$$

以 Python 實現，竟然只要兩行代碼：
```python
def gcd(a, b):
    return a if b == 0 else gcd(b, a % b)
```
- **極速時間複雜度**：每經過兩回合，數值至少減半！時間複雜度只有驚人的 $O(\log(\min(a, b)))$。即使 $a, b$ 高達 100 位數，電腦也能在千分之一秒內算完！
- **自動修正大小順序**：就算呼叫時小數在前 `gcd(18, 48)`，第一輪計算 `18 % 48 = 18`，下一層就會變成 `gcd(48, 18)`，自動幫你換位完成！

#### 3. 初學者常見陷阱：Base Case 判定寫錯
初學同學常誤把 Base Case 寫成：`if a == 0: return b`，或者在餘數為 0 時不知道該回傳誰。
請牢記口訣：**「除數 $b$ 變成 0 之時，被除數 $a$ 就是最大公因數！」**
因此 Base Case 永遠是：`if b == 0: return a`。

#### 4. APCS 實戰視野
最大公因數（GCD）是 APCS 實作題數論題型的必備武器：
- **分數約分化簡**：分子分母同除以 $\gcd(\text{分子}, \text{分母})$。
- **最小公倍數（LCM）**：公式為 $\text{LCM}(a, b) = (a \times b) // \gcd(a, b)$。掌握這兩行遞迴，考場數論題迎刃而解！

In [ ]:
# 範例 11.7.4：歐幾里得輾轉相除法遞迴追蹤

def gcd(a, b):
    print(f"  [呼叫 gcd] a = {a:3d}, b = {b:3d}")
    # Base Case: 當 b 縮小為 0 時，a 即為最大公因數！
    if b == 0:
        print(f"    ⭐ 餘數歸零！最大公因數為: {a}")
        return a
    # Recursive Case: 傳入 (b, a % b)
    return gcd(b, a % b)

def lcm(a, b):
    """利用 GCD 計算最小公倍數 LCM"""
    return (a * b) // gcd(a, b)

print("=== 求解 gcd(48, 18) 追蹤 ===")
ans_gcd = gcd(48, 18)
print(f"gcd(48, 18) = {ans_gcd}\n")

print("=== 求解 lcm(12, 18) ===")
ans_lcm = lcm(12, 18)
print(f"lcm(12, 18) = {ans_lcm}")

In [ ]:
# ==========================================
# [3] Code 填空題 11.7.4
# 任務說明：
# 某位同學正在實作極簡的一行流 gcd 函數。
# 請補齊程式碼中的 `___`：
# 1. 判斷 b 是否為 0。
# 2. 若不為 0，遞迴呼叫 (b, a % b)。
# ==========================================

def fast_gcd(a, b):
    # 提示：三元運算子，b 為 0 則回傳 a，否則遞迴
    return a if b == ___ else fast_gcd(b, a % ___)

# 主程式測試
res1 = fast_gcd(54, 24)
res2 = fast_gcd(17, 13)
print(f"gcd(54, 24) = {res1}")  # 預期輸出: 6
print(f"gcd(17, 13) = {res2}")  # 預期輸出: 1 (互質)

In [ ]:
# ==========================================
# [4] Code 練習題 11.7.4
# 任務說明：
# 請設計一個「分數約分化簡器」`simplify_fraction(num, den)`：
# 1. 接收分子 num 與分母 den（均為正整數）。
# 2. 使用遞迴 gcd(a, b) 計算分子分母的最大公因數 g。
# 3. 將兩者同除以 g 進行約分。
# 4. 回傳字串格式 f"{num//g}/{den//g}"，若分母約分後為 1，則直接回傳純整數字串 f"{num//g}"。
#
# 【公開測試資料 1】
# 呼叫：simplify_fraction(18, 24)
# 預期輸出：
# 約分結果: 3/4
#
# 【公開測試資料 2】
# 呼叫：simplify_fraction(35, 7)
# 預期輸出：
# 約分結果: 5
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.7.4
# 任務說明：
# 請設計一個「三個整數的最大公因數」計算函數 `gcd_three(a, b, c)`：
# 1. 數學結合律原理：gcd(a, b, c) = gcd(gcd(a, b), c)。
# 2. 先利用遞迴求出 a 與 b 的最大公因數 g1。
# 3. 再將 g1 與 c 呼叫遞迴求出最終最大公因數。
# 4. 驗證 gcd_three(24, 36, 60) 是否正確輸出 12。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

### 11.7.5 快速冪運算遞迴版：將 $O(N)$ 降至 $O(\log N)$ 的分治思維萌芽

#### 1. 生活故事比喻：一張紙的對摺奇蹟
想像你手中拿著一張大紙，你的目標是要製造出 16 張厚度的紙疊（計算 $a^{16}$）。
- **笨方法（線性累乘）**：一張一張拿起來疊，要重複疊 16 次（時間複雜度 $O(N)$）；
- **聰明方法（對摺倍增）**：
  你把紙對摺一次，厚度變 2 倍；
  再對摺一次，厚度變 4 倍；
  再對摺一次，厚度變 8 倍；
  再對摺第四次，厚度立刻達到 **16 倍**！
只要短短「4 次對摺」，就完成了需要 16 次的繁瑣工作！這就是演算法中赫赫有名的**「快速冪（Binary Exponentiation）」**！

#### 2. 底層運作機制：分治法（Divide and Conquer）的對稱分解
計算 $a^b$ 時，依據指數 $b$ 是偶數還是奇數，可以拆解為：
$$a^b = \begin{cases} 1 & \text{if } b = 0 \quad (\text{Base Case}) \\ (a^{b // 2})^2 & \text{if } b \text{ 為偶數（對摺！）} \\ a \times a^{b - 1} & \text{if } b \text{ 為奇數（抽出一個變偶數後再對摺）} \end{cases}$$

例如計算 $2^{10}$：
- $10$ 是偶數 $\rightarrow$ 先算 $2^5$，算好後平方！
- $5$ 是奇數 $\rightarrow$ 轉成 $2 \times 2^4$！
- $4$ 是偶數 $\rightarrow$ 先算 $2^2$ 平方！
- $2$ 是偶數 $\rightarrow$ 先算 $2^1$ 平方！
- $1$ 是奇數 $\rightarrow$ $2 \times 2^0$！
- $0$ 觸發 Base Case 回傳 1！
原本需要連乘 10 次的運算，縮減為僅僅 5 步！若計算 $2^{1000000000}$（10 億次方），傳統迴圈需要算 10 億次（必然超時 TLE），快速冪只要約 **30 次** 就能瞬間算完！時間複雜度從 $O(N)$ 暴降至 $O(\log N)$！

#### 3. 初學者常見陷阱：平方時寫錯引發退化
初學同學在寫偶數平方時，常順手寫成：
```python
return fast_pow(a, b // 2) * fast_pow(a, b // 2)  # 致命失誤！
```
請特別警惕！如果你寫了兩次 `fast_pow`，直譯器就會把左邊算一遍、右邊又算一遍，硬生生把一條線拆成了一棵巨大二元樹，時間複雜度立刻退化回 $O(N)$ 甚至更慢！
正確寫法是：**先用一個區域變數接住答案，再將其自乘**：
```python
half = fast_pow(a, b // 2)
return half * half  # 只算一次，極速 O(log N)！
```

#### 4. APCS 實戰視野
快速冪是「分治思維」最完美的啟蒙範例。在後續的大數模運算、密碼學 RSA、矩陣快速冪求極速費氏數列等 APCS 高級題型中，快速冪是選手必備的核心武功。

In [ ]:
# 範例 11.7.5：遞迴版快速冪演算法實作與乘法次數統計

mul_count = 0

def fast_power(a, b):
    global mul_count
    # Base Case: 任何數的 0 次方均為 1
    if b == 0:
        return 1
        
    # 若次方為奇數：提出一個 a，轉化為偶數次方
    if b % 2 != 0:
        mul_count += 1
        return a * fast_power(a, b - 1)
        
    # 若次方為偶數：對摺！先計算一半次方
    half = fast_power(a, b // 2)
    mul_count += 1
    return half * half  # ⭐ 關鍵：只算一次 half，再相乘！

print("=== 快速冪計算 2^16 ===")
mul_count = 0
ans = fast_power(2, 16)
print(f"2^16 = {ans}")
print(f"傳統連乘需要 16 次乘法，快速冪僅耗費: {mul_count} 次乘法！")

print("\n=== 快速冪計算 3^13 ===")
mul_count = 0
ans_odd = fast_power(3, 13)
print(f"3^13 = {ans_odd} (僅耗費 {mul_count} 次乘法)")

In [ ]:
# ==========================================
# [3] Code 填空題 11.7.5
# 任務說明：
# 某位同學正在實作遞迴快速冪函數 `rec_pow(base, exp)`。
# 請補齊程式碼中的 `___`：
# 1. Base Case：exp == 0 時回傳 1。
# 2. 偶數次方時呼叫 `exp // 2` 並計算 `half * half`。
# ==========================================

def rec_pow(base, exp):
    # 提示：0 次方等於 1
    if exp == 0:
        return ___
        
    if exp % 2 == 1:
        # 奇數次方
        return base * rec_pow(base, exp - 1)
    else:
        # 偶數次方：先算一半
        half = rec_pow(base, exp // ___)
        return half * ___

# 主程式測試 5^4 (預期 625)
val = rec_pow(5, 4)
print(f"5^4 = {val}")  # 預期輸出: 625

In [ ]:
# ==========================================
# [4] Code 練習題 11.7.5
# 任務說明：
# 在競賽中，大數次方常需要對一個大質數取餘數（如 1000000007），防止數字過大。
# 請設計「取模快速冪」函數 `modular_pow(a, b, m)`：
# 1. 計算 (a^b) % m。
# 2. Base Case：b == 0 時回傳 1 % m。
# 3. 奇數時：`((a % m) * modular_pow(a, b - 1, m)) % m`。
# 4. 偶數時：`half = modular_pow(a, b // 2, m)`，回傳 `(half * half) % m`。
#
# 【公開測試資料 1】
# 呼叫：modular_pow(2, 10, 1000)
# 預期輸出：
# 2^10 % 1000 = 24  (1024 % 1000)
#
# 【公開測試資料 2】
# 呼叫：modular_pow(3, 7, 100)
# 預期輸出：
# 3^7 % 100 = 87  (2187 % 100)
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.7.5
# 任務說明：
# 請設計一個對比實驗，比較計算 2^100 時：
# 1. 傳統累乘迴圈（每次乘 2，重複 100 次）。
# 2. 遞迴快速冪（對摺分治）。
# 3. 分別統計兩者實際執行的「乘法運算次數」，印出詳細數據並證明分治法帶來的幾何級數加速效果。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

### 11.7.6 遞迴思維總結：何時該用迴圈（迭代）？何時遞迴思維更直觀？

#### 1. 生活故事比喻：平坦公路的自行車 vs 險峻峽谷的立體直升機
- **迴圈（Iteration）**：就像一輛輕盈耐操的公路自行車。沿著筆直平坦的環島公路前進時，它完全不耗油（空間複雜度 $O(1)$，絕不爆棧），踩幾萬公里也穩如泰山。如果只是單純的「把陣列看一遍」、「數 1 到 100 萬的總和」，騎自行車是最明智、最經濟的選擇。
- **遞迴（Recursion）**：就像一架具備立體機動能力的空中直升機。當你面對的是曲折分岔的喀斯特溶洞、盤根錯節的原始樹冠、或者多重嵌套的分治地圖時，自行車寸步難行，而直升機卻能輕鬆升降、左右分飛、原路返航！

#### 2. 底層運作機制：遞迴與迴圈的全維度評估對照
在編寫 APCS 程式時，優秀選手大腦中常備這張決策雷達圖：

| 評估維度 | 迴圈（迭代 Iteration） | 遞迴（Recursion） |
| :--- | :--- | :--- |
| **空間消耗** | 極致省電 $O(1)$（原地更新變數） | 需佔用 $O(\text{深度})$ 的 Call Stack 記憶體 |
| **棧溢位風險** | 零風險（跑 1000 萬次也絕不爆棧） | 深度超過 1000 層引爆 `RecursionError` |
| **程式碼直觀度** | 處理單線線性問題極為自然 | 處理「樹狀走訪、分治、回溯」代碼量少 70% |
| **效能開銷** | 無額外開銷，直譯執行速度最快 | 每層呼叫有 Frame 建立與銷毀之微小開銷 |

#### 3. 初學者常見陷阱：兩大極端偏見
- **極端一：遞迴萬能論**  
  把簡單的累加、找最大值硬要寫成遞迴，不僅程式碼晦澀難懂，更隨時面臨 1000 層爆棧（Runtime Error）的巨大風險。
- **極端二：遞迴恐懼症**  
  因為害怕遞迴，遇到二元樹遍歷、全排列生成、走迷宮深度搜尋時，硬要用陣列手刻模擬堆疊，寫出兩百行冗長又極易出錯的代碼。

#### 4. APCS 實戰視野：競賽決策三大黃金法則
1. **問題是線性前進的嗎？**（例如：計數、過濾、尋找串列極值）  
   $\rightarrow$ **堅決使用 `for` 或 `while` 迴圈！**
2. **問題深度是否受限於對數級？**（如二分搜尋、GCD、快速冪，深度 $\le 30$）  
   $\rightarrow$ **果斷使用遞迴，程式碼精煉優雅無負擔！**
3. **問題本質是樹狀或回溯嗎？**（如迷宮路徑探索、二元樹走訪、八皇后問題）  
   $\rightarrow$ **遞迴是唯一通往滿分 AC 的康莊大道！**

In [ ]:
# 範例 11.7.6：同一個問題（計算數列階乘）的雙思維代碼對照

def factorial_loop(n):
    """迴圈版本：空間複雜度 O(1)，絕無爆棧風險"""
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result

def factorial_rec(n):
    """遞迴版本：高度符合數學歸納法，空間複雜度 O(n)"""
    return 1 if n <= 1 else n * factorial_rec(n - 1)

test_val = 6
print(f"迴圈版本計算 {test_val}! = {factorial_loop(test_val)}")
print(f"遞迴版本計算 {test_val}! = {factorial_rec(test_val)}")
print("結論：兩者數學本質相通，依據問題結構與深度靈活挑選最佳工具！")

In [ ]:
# ==========================================
# [3] Code 填空題 11.7.6
# 任務說明：
# 某位同學正在整理遞迴與迴圈的選型思維。
# 請補齊程式碼中的 `___`，完成兩種方式計算平方和的實作：
# 1. 迴圈版本累加。
# 2. 遞迴版本累加。
# ==========================================

def square_sum_loop(n):
    total = 0
    for i in range(1, n + 1):
        total += i * ___  # 提示：累加平方
    return total

def square_sum_rec(n):
    if n <= 0:
        return 0
    # 提示：當前項平方加上前 n-1 項遞迴
    return (n * n) + square_sum_rec(___)

# 主程式測試 1^2 + 2^2 + 3^2 = 14
print("迴圈計算平方和:", square_sum_loop(3))  # 預期: 14
print("遞迴計算平方和:", square_sum_rec(3))   # 預期: 14

In [ ]:
# ==========================================
# [4] Code 練習題 11.7.6
# 任務說明：
# 請設計一個「二分法折半查詢次數」計算器：
# 給定一個已排序陣列長度 N，每次折半（N // 2），直到 N 降為 1。
# 請分別用：
# 1. `count_steps_loop(n)`：使用 while 迴圈計算需要折半幾次。
# 2. `count_steps_rec(n)`：使用遞迴計算需要折半幾次。
# 驗證兩者回傳結果完全一致。
#
# 【公開測試資料 1】
# 呼叫：計算 N = 16 的折半次數
# 預期輸出：
# 迴圈折半次數: 4
# 遞迴折半次數: 4
#
# 【公開測試資料 2】
# 呼叫：計算 N = 100 的折半次數
# 預期輸出：
# 迴圈折半次數: 6
# 遞迴折半次數: 6
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.7.6
# 任務說明：
# 請設計一個演算法選型診斷器 `algorithm_advisor(problem_type, data_size)`：
# 1. 若 problem_type == "linear"（線性走訪、加總）：
#    - 若 data_size > 1000，回傳 "強烈建議使用迴圈（防止 RecursionError 爆棧）"。
#    - 否則回傳 "迴圈與遞迴皆可，優先推薦迴圈"。
# 2. 若 problem_type == "tree"（二元樹走訪、全排列枚舉）：
#    - 回傳 "強烈推薦使用遞迴（程式碼極簡且思維直觀）"。
# 3. 若 problem_type == "divide_and_conquer"（分治、二分法、快速冪）：
#    - 回傳 "極度推薦使用遞迴（深度僅為 log N，安全且高效）"。
# 4. 在主程式中測試多種情境，並印出評估診斷結果。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

## 11.7 單元重點回顧與自我檢驗

恭喜你完成了第 11.7 單元的深度探索！從單線遞迴跨入樹狀分支遞迴，你的程式思維已經成功突破了一維線性的束縛，邁向了二維樹狀展開的壯麗景致。

### 核心觀念複習清單
1. **雙分支遞迴模型**：
   - 函數內部包含多次自我呼叫，執行軌跡形成樹狀結構。
   - 單執行緒 Python 嚴格遵循「深度優先（Depth-First）」時序：左子樹徹底執行完畢並彈出後，才展開右子樹。
2. **經典費氏數列（Fibonacci）**：
   - $F(n) = F(n-1) + F(n-2)$ 必須同時具備 $n=0$ 與 $n=1$ 雙重 Base Case，防止邊界錯位。
3. **重疊子問題與指數級爆炸**：
   - 樸素樹狀遞迴時間複雜度高達 $O(2^n)$，重複計算帶來嚴重的時間浩劫。
   - 體會演算法時間複雜度的巨大震撼，為後續動態規劃（記憶化查表 Memoization）鋪平道路。
4. **歐幾里得輾轉相除法（GCD）**：
   - 短短兩行遞迴神技：`a if b == 0 else gcd(b, a % b)`。
   - $O(\log(\min(a, b)))$ 極速數論武器，解鎖分數約分與最小公倍數（LCM）。
5. **快速冪運算的分治思維**：
   - 摺紙對摺模型：偶數次方先算一半 `half` 再相乘，將 $O(N)$ 降至 $O(\log N)$。
6. **遞迴與迴圈的理性抉擇**：
   - 線性大數據堅決用迴圈（空間 $O(1)$，防爆棧）；分治與樹狀問題果斷用遞迴（優雅直觀）。

---

### 下一步精彩預告
在即將到來的第十一章壓軸單元，我們要把所有學過的函數語法、作用域原則、輔助工具以及模組化思想融會貫通！
- 如何運用「自頂向下（Top-Down）拆題法」，將 APCS 複雜的大題目拆成 30 行以內的精煉代碼？
- 競賽中必備的「二維網格安全邊界檢查」、「曼哈頓距離」、「數論判定」三大黃金輔助函式庫該如何建立？
- 如何進行「單元測試隔離除錯」，在考場緊張的倒數計時中 3 秒鐘揪出隱藏 Bug？

請緊接著邁向 **[11.8 函數模組化解題實戰：Top-Down 拆題法與競賽輔助函數庫建立]**，完成第十一章的大圓滿完工！